hours=6 was the only misprediction (that's what "matched exactly row-for-row" except for that one case meant yesterday), the real data for 10 students (hours 1→10, index 0→9) is:
```
predictions: [0 0 0 0 0 1 1 1 1 1]
actual:      [0 0 0 0 0 0 1 1 1 1]   ← only index 5 (hours=6) differs
index:        0 1 2 3 4 5 6 7 8 9
```
Walking it with you, corrected:
* Index 0-4: predicted=0, actual=0 → TN (5 of these)
* Index 5 (hours=6): predicted=1, actual=0 → FP (1 of these — your false alarm)
* Index 6-9: predicted=1, actual=1 → TP (4 of these)
* FN count: 0 — the model never once said "fail" when the student actually passed

So the real confusion matrix is:
```
                    Actual: PASS (1)    Actual: FAIL (0)
Predicted: PASS(1)  TP = 4              FP = 1
Predicted: FAIL(0)  FN = 0              TN = 5

Total = 4+1+0+5 = 10 ✓ matches your 10 students.
```
Now — here's where it gets useful. Accuracy = (correct predictions) / (total) = (TP+TN)/(TP+TN+FP+FN) = 9/10 = 90%.

But accuracy treats all 4 boxes as if only "right vs. wrong" mattered — it can't tell you whether your model tends to false-alarm or tends to miss things.

**Precision** asks: "Of everything I predicted PASS, how many actually passed?"
```
Precision = TP / (TP + FP)
```
**Recall** asks: "Of everything that actually passed, how many did I catch?"
```
Recall = TP / (TP + FN)
```
Using your matrix (TP=4, FP=1, FN=0)

That's the F1-score — the harmonic mean of precision and recall:
```
F1 = 2 × (Precision × Recall) / (Precision + Recall)
```

In [2]:
import numpy as np

y_actual = np.array([0, 0, 0, 0, 0, 0, 1, 1, 1, 1])
y_pred   = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])

print("actual:", y_actual)
print("pred:  ", y_pred)

actual: [0 0 0 0 0 0 1 1 1 1]
pred:   [0 0 0 0 0 1 1 1 1 1]


In [3]:
# TP = Predicted = 1 AND actual = 0
TP = np.sum((y_pred == 1) & (y_actual == 1))
print("TP:", TP)

TP: 4


In [4]:
# FP = predicted=1 AND actual=0
FP = np.sum((y_pred == 1) & (y_actual == 0))
print("FP:", FP)

FP: 1


In [5]:
# FN = predicted=0 AND actual=1
FN = np.sum((y_pred == 0) & (y_actual == 1))
print("FN:", FN)

FN: 0


In [6]:
# TN = predicted = 0 AND actual = 0
TN = np.sum((y_pred == 0) & (y_actual == 0))
print("TN:", TN)

TN: 5


In [7]:
precision = TP / (TP + FP)
recall = TP / (TP + FN)
f1 = 2 * (precision * recall) / (precision + recall)

print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

Precision: 0.8
Recall: 1.0
F1: 0.888888888888889


In [8]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

print("sklearn confusion matrix:\n", confusion_matrix(y_actual, y_pred))
print("sklearn precision:", precision_score(y_actual, y_pred))
print("sklearn recall:", recall_score(y_actual, y_pred))
print("sklearn f1:", f1_score(y_actual, y_pred))

sklearn confusion matrix:
 [[5 1]
 [0 4]]
sklearn precision: 0.8
sklearn recall: 1.0
sklearn f1: 0.8888888888888888


Day 10 — Session Summary:

* Built the Confusion Matrix (TP/FP/FN/TN) from first principles using yesterday's own model's mistake as the motivating example
* Derived Precision and Recall from the same 4 boxes, and the mechanical tradeoff between them (verified by reasoning through what a lower threshold does to FN→TP conversion)
* Derived F1 as harmonic mean, and why it punishes imbalance (via the extreme P=1.00/R=0.01 example)
* Built confusion matrix + all 3 metrics from scratch in NumPy, verified exact match against sklearn
* Decoded sklearn's actual/predicted axis-and-order convention — a real gotcha
* Applied the Precision vs. Recall decision to a live scenario (medical screening) and correctly reasoned through the cost asymmetry that determines which metric to prioritize